# Level 3: branching fed-batch reconciliation
Select compatible, conflicting, switching, growth or nonunique input/output paths. Run the six forward-only cells.


In [1]:
# STEP 1: Select folders
# WHAT TO DO: Run with Shift+Enter, then continue in order, or use Run All. Change example_dir
# for a copied case and EXPORT_DIR to preserve existing outputs.

from pathlib import Path
import csv
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "PharmaPy").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from PharmaPy.Bioreactors import build_bioreactor, BioreactorUnitConverter as Units

example_dir = repo_root / "examples/bioreactors/generic_fed_batch/reconciliation/level3"
input_dir = example_dir / "inputs/switching"
EXPORT_DIR = example_dir / "outputs/switching"
WRITE_ARTIFACTS = True

In [2]:
# STEP 2: Load inputs
# WHAT TO DO: Edit the JSON files, then rerun this and subsequent cells. See README.md for
# the source-to-JSON unit conversions; no fitting is performed.

case = json.loads((input_dir / "case.json").read_text())
config = json.loads((input_dir / "mechanism.json").read_text())
thermo_path = input_dir / "thermo.json"


In [3]:
# STEP 3: Run the native forward simulation
# WHAT TO DO: Run to build a fresh reactor from the three input files and solve its recipe.
# For another supported model, change input_dir in Step 1; no local adapters are required.
assembly = build_bioreactor(case, config, thermo_path)
histories = assembly.solve(verbose=False)
phase = assembly.phase
print("Native reactor:", type(assembly.unit).__name__)


Native reactor: SemiBatchReactor


In [4]:
# STEP 4: Build the trajectory table
# WHAT TO DO: Run to report all non-carrier species and the selected model's biological states.
# Species use g and g/L; biological states retain declared units, except kg becomes g.
# Extensive biological states also receive per-L columns; event-side rows remain separate.
species = [name for name in phase.name_species if name != assembly.carrier_species]
columns = {name+'_g_l': name+' (g/L)' for name in species}
rows = []
for result in histories:
    for i, time in enumerate(result.time):
        volume_l = Units.convert(np.asarray(result.vessel_vol).reshape(-1)[i], 'm3', 'L')
        row = dict(time_h=Units.convert(time, 's', 'h'), volume_l=volume_l)
        mass_g = Units.convert(result.mass_j_liquid0[i], 'kg', 'g')
        for name in species:
            mass = float(mass_g[list(phase.name_species).index(name)])
            row.update({name+'_g': mass, name+'_g_l': mass/volume_l})
        for state in assembly.mechanism.solver_states:
            value = float(np.asarray(getattr(result, state.name+'_liquid0')[i]).item())
            key, unit = state.name, state.units
            if unit in {'kg', 'kgDW'}:
                value = Units.convert(value, 'kg', 'g')
                key, unit = key.removesuffix('_kg')+'_g', 'g'
            row[key] = value
            if state.name in getattr(assembly.mechanism, 'extensive_states', ()):
                key = key+'_l' if key.endswith('_g') else key+'_per_l'
                row[key] = value/volume_l
                unit += '/L'
            columns[key] = key+' ('+unit+')'
        rows.append(row)
columns['volume_l'] = 'Volume (L)'
print("Final row:", rows[-1])


Final row: {'time_h': 24.0, 'volume_l': 1.1000000000000132, 'nutrient_a_g': 6.271602591888203e-19, 'nutrient_a_g_l': 5.70145690171648e-19, 'nutrient_b_g': 0.0, 'nutrient_b_g_l': 0.0, 'byproduct_g': 0.05489361702127659, 'byproduct_g_l': 0.04990328820115994, 'viable_cells_million': 0.9148936170212774, 'viable_cells_million_per_l': 0.831721470019333, 'dead_cells_million': 0.0, 'dead_cells_million_per_l': 0.0, 'product_g': 0.08234042553191509, 'product_g_l': 0.07485493230174009, 'ivcd_million_cell_day_per_ml': 0.0009135070191259311}


In [5]:
# STEP 5: Plot the simulated trajectories
# WHAT TO DO: Run to plot the quantities discovered in Step 4; optionally edit columns here.
# Species names, model states and the plot title come from the selected inputs.
figure, axes = plt.subplots(int(np.ceil(len(columns)/2)), 2, figsize=(10, 3.5*np.ceil(len(columns)/2)),
                            squeeze=False, constrained_layout=True)
time_h = [row['time_h'] for row in rows]
for axis, (key, label) in zip(axes.flat, columns.items()):
    axis.plot(time_h, [row[key] for row in rows])
    axis.set(xlabel='Time (h)', ylabel=label)
    axis.grid(alpha=.25)
for axis in list(axes.flat)[len(columns):]:
    axis.set_visible(False)
figure.suptitle(case['recipe']['name']+' — native forward simulation')
plt.show()


In [6]:
# STEP 6: Export spreadsheet and figures
# WHAT TO DO: Run to export CSV (opens in spreadsheet software), PNG and SVG. Set
# WRITE_ARTIFACTS=False in Step 1 to run without replacing saved outputs.

if WRITE_ARTIFACTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    with (EXPORT_DIR / "trajectories.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    figure.savefig(EXPORT_DIR / "trajectories.png", dpi=180)
    figure.savefig(EXPORT_DIR / "trajectories.svg")
    print("Saved forward trajectories and figures to", EXPORT_DIR)

Saved forward trajectories and figures to examples/bioreactors/generic_fed_batch/reconciliation/level3/outputs/switching
